# Generative unsupervised models — plain-language explanation

A **generative unsupervised model** studies examples that do not have labels and tries to learn the patterns they have in common. Its goal is to create new examples that look as though they could have come from the same dataset.

For example, suppose the training data contains thousands of photographs of human faces. The model learns that faces usually have two eyes, a nose below them, and certain ranges of colors, shapes, and proportions. It can then generate a face that was not in the training set but still looks like a plausible human face.

## What is a probability distribution here?

A **probability distribution** is a rule that says how likely different possible examples are. For a model trained on faces:

- realistic face-like images should receive high probability;
- unusual but possible faces might receive lower probability;
- random television-like noise should receive extremely low probability.

An **explicit generative model** learns a mathematical description of this distribution, often written as $p(x)$, where $x$ is a possible data example. **Sampling from the distribution** means asking the model to make a random draw: common patterns are likely to appear, while rare patterns appear less often. Each draw can therefore produce a different new example.

A useful analogy is a loaded die. Its probability distribution describes how likely each side is to appear. Sampling means rolling the die. A generative model works with much more complicated outcomes—such as every possible image—instead of only six sides.

## What does statistically indistinguishable mean?

It does **not** mean that a generated example must copy a training example. It means that, ideally, a large collection of generated examples has the same measurable patterns as the real data. If the model works well, it should be difficult to decide whether an example is real or generated by looking at those patterns alone. In practice, models only approximate this goal and may still produce unrealistic examples.

## In one sentence

The model learns which kinds of data examples are common or rare, then uses that learned rule to randomly create new examples that resemble the training data without simply reproducing it.


# Latent variables, latent space, and VAEs

## Short answer

**Yes—this is directly related to the latent space in a variational autoencoder (VAE).** A VAE is one particular model that implements this idea. The idea is broader than VAEs, however: GANs and several other generative models also generate data from latent variables, while some generative models do not use a lower-dimensional latent space at all.

## Why can high-dimensional data have a lower-dimensional structure?

A $256 \times 256$ RGB image contains $256 \times 256 \times 3 = 196{,}608$ observed numbers. Mathematically, every one of those numbers could vary independently, so the space of all possible images is enormous. But almost every random arrangement of those pixels looks like noise, not like a photograph.

Real photographs are constrained by how the world works. Objects have coherent shapes, nearby pixels are related, light creates predictable shadows, and cameras project a three-dimensional scene onto a two-dimensional image. Consequently, real images occupy only a small, structured region of the space of all possible pixel arrangements. This is the intuition behind saying that the data has a lower **intrinsic dimension** than its raw number of observed variables suggests. This structured region is often informally called the **data manifold**.

The sentence example makes the same point. A sequence of 20 words has a huge number of possible arrangements, but grammar and meaning rule out nearly all of them. Meaningful sentences therefore form a small, structured subset of all random word sequences.

## What is a latent variable?

A **latent variable** is an underlying value that helps explain an observation but is not directly present as one of the observed measurements. For a face image, useful underlying factors might include identity, head angle, expression, lighting, hairstyle, and background. The model is not necessarily given these concepts or guaranteed to represent them in such a clean, human-interpretable way; they are simply an intuition for what latent variables can encode.

We can write the observed data as $x \in \mathbb{R}^D$ and its latent representation as $z \in \mathbb{R}^d$, where usually $d \ll D$. The set of possible values of $z$ is the **latent space**. A neural network learns a mapping from latent space to data space:

$$z \longrightarrow f_\theta(z) \approx x$$

Here, $f_\theta$ is often called a **decoder** or **generator**. It turns a compact latent description into all the pixels, words, or other observed values of a complete example.

## How does this generate new data?

Instead of defining a complicated probability distribution directly over every possible image, the model starts with a simple distribution in latent space, commonly

$$z \sim \mathcal{N}(0, I).$$

Generating an example then has two steps:

1. Randomly sample a latent point $z$ from the simple distribution.
2. Pass $z$ through the decoder or generator to obtain $x = f_\theta(z)$.

The neural network performs the difficult part: it bends and expands the simple latent distribution into the complicated distribution of realistic data. Different sampled points produce different examples.

A useful analogy is a compact set of instructions for a scene. The latent point says something like ‘this identity, this pose, this expression, and this lighting,’ while the decoder converts those compact instructions into hundreds of thousands of pixel values. Again, an actual model may organize its latent variables differently and less interpretably.

## Why does interpolation work?

Suppose two real examples have latent representations $z_A$ and $z_B$. An intermediate latent point can be formed with

$$z(t) = (1-t)z_A + tz_B, \qquad 0 \leq t \leq 1.$$

Decoding $z(t)$ as $t$ moves from 0 to 1 can produce a smooth sequence between the examples. This works when nearby points in the learned latent space decode to similar, plausible data. Interpolating the raw pixels instead often just creates a transparent-looking blend, whereas interpolation in a well-organized latent space can gradually change higher-level properties such as pose or expression.

## How a VAE uses this idea

A VAE contains two main neural networks:

- The **encoder** takes an example $x$ and produces a probability distribution for its likely latent representation, written $q_\phi(z \mid x)$.
- The **decoder** takes a sampled latent value $z$ and describes how to reconstruct or generate an example, written $p_\theta(x \mid z)$.

During training, the VAE learns both to reconstruct data and to keep latent representations close to a simple prior distribution, usually $p(z)=\mathcal{N}(0,I)$. That regularized organization is what lets us later sample a new $z$ from the prior and decode it into a plausible new example. It also encourages useful interpolation between latent points.

So the generation path in a VAE is:

$$z \sim p(z) \quad\longrightarrow\quad x \sim p_\theta(x \mid z).$$

The encoder is needed to learn representations of existing examples; generation from scratch only needs a sampled latent point and the decoder. Chapter 17 develops this in detail: [Variational autoencoders](../../17-variational-autoencoders/README.md).

## Why the book says ‘some, but not all’

A lower-dimensional latent space is one design choice, not a requirement for every generative model. VAEs and standard GANs commonly use latent variables. Autoregressive models can generate one value at a time without a compact latent bottleneck. Normalizing flows commonly use latent and data spaces of the same dimension so the mapping remains invertible. Standard diffusion models usually begin with noise having the same dimensions as the data, although **latent diffusion** models first compress data and then run diffusion in a lower-dimensional latent space.

## Mental model

Think of the full data space as every pixel arrangement that could exist. Real data occupies a much smaller structured region. A latent space provides compact coordinates for navigating that region, and a decoder translates those coordinates into complete observations. A VAE learns both how to place existing examples into those coordinates and how to map coordinates back into data.
